In [1]:
import sys
sys.path.insert(0, "/Users/aidavillalba/Desktop/Proyectos/2026/02_Libreria_multimodal/multimodalpy")
from multimodalpy import main
import os
import numpy as np
import geopandas as gpd
import seaborn as sns
import pandas as pd
from shapely import wkt
import shapely

from shapely.errors import ShapelyError
from dataclasses import dataclass, field

NAP_API_KEY= "44a2e8d8-fab3-4d87-940d-d7788a7dac94"

output_folder = "/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia"

### Functions

In [2]:
NODE_COLS_OSM = ["node_id", "node_role", "geometry"]
EDGE_COLS_OSM = ["edge_id", "from_node_id", "to_node_id", "highway", "lanes",
             "maxspeed", "name", "oneway", "reversed", "length", "tts",
             "geometry"]
 
VALID_NODE_ROLES_OSM = {"intersection", "linear_vertex", "through_endpoint", "endpoint"}
 
# base OSM highway tags; compound values ("a;b") are checked part-by-part
KNOWN_HIGHWAY_TAGS = {
    "motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link", "cycleway", "footway",
    "secondary", "secondary_link", "tertiary", "tertiary_link", "unclassified",
    "residential", "living_street", "service", "busway", "pedestrian", "track",
    "road",
}


NODE_COLS_GTFS = ["node_id", "stop_name", "geometry"]
EDGE_COLS_GTFS = [
    "edge_id", "from_node_id", "to_node_id", "route_id", "route_short_name",
    "route_long_name", "trip_count", "tts_mean", "tts_min", "tts_max",
    "tts_mean_peak_am", "tts_mean_peak_pm", "tts_mean_rest_of_day",
    "trip_count_peak_am", "trip_count_peak_pm", "trip_count_rest_of_day",
    "tts_mean_weekday", "trip_count_weekday", "tts_mean_weekend",
    "trip_count_weekend", "days_of_week_summary", "hourly_travel_times",
    "hourly_trip_counts", "tts_mean_peak_am_weekday", "trip_count_peak_am_weekday",
    "tts_mean_peak_am_weekend", "trip_count_peak_am_weekend",
    "tts_mean_peak_pm_weekday", "trip_count_peak_pm_weekday",
    "tts_mean_peak_pm_weekend", "trip_count_peak_pm_weekend",
    "tts_mean_rest_of_day_weekday", "trip_count_rest_of_day_weekday",
    "tts_mean_rest_of_day_weekend", "trip_count_rest_of_day_weekend", "geometry",
]
 
VALID_DAYS = {"monday", "tuesday", "wednesday", "thursday", "friday",
              "saturday", "sunday"}

 
COORD_TOL = 0.01     # metres: geometry endpoints vs node coords, in projected CRS
LENGTH_REL_TOL = 0.02  # 2% relative tolerance between `length` col and geometry length

In [3]:
@dataclass
class CheckResult:
    name: str
    level: str          # "fail" | "warn"
    n_issues: int
    issues: pd.DataFrame | None = None
    note: str = ""
 
 
def _fmt(r: CheckResult) -> str:
    tag = "OK" if r.n_issues == 0 else f"{r.level.upper()} ({r.n_issues})"
    return f"[{tag}] {r.name}" + (f" — {r.note}" if r.note else "")
 
 
def _record(results, name, level, subset, note=""):
    r = CheckResult(name=name, level=level, n_issues=len(subset),
                     issues=subset if len(subset) else None, note=note)
    results.append(r)
    print(_fmt(r))
    if r.n_issues and r.issues is not None:
        print(r.issues.head(5).to_string())
        print()
    return r
 
 
def check_schema_osm(df, expected_cols, name, results):
    print(f"--- {name}: schema ---")
    missing = set(expected_cols) - set(df.columns)
    extra = set(df.columns) - set(expected_cols)
    _record(results, f"{name}: expected columns present", "fail",
            pd.Index(sorted(missing)))
    if extra:
        print(f"[INFO] {name}: extra/unexpected columns: {sorted(extra)}")
    print()
 
 
def _parse_geometries(geom_series: pd.Series) -> np.ndarray:
    """
    Return a numpy array of shapely geometries from a column that may hold
    either WKT strings (plain pandas DataFrame / CSV read) or already-parsed
    shapely geometry objects (e.g. a GeoDataFrame's geometry column).
    Malformed/missing entries become None.
    """
    values = geom_series.to_numpy()
    is_str = np.array([isinstance(v, str) for v in values])
 
    out = np.empty(len(values), dtype=object)
    if is_str.any():
        str_vals = np.where(is_str, values, "")
        parsed = shapely.from_wkt(str_vals, on_invalid="ignore")
        out[is_str] = parsed[is_str]
 
    not_str = ~is_str
    if not_str.any():
        for i in np.where(not_str)[0]:
            v = values[i]
            out[i] = v if isinstance(v, shapely.Geometry) else None
 
    return out
 
 
def _multi_value_bad_mask(series: pd.Series, positive_only: bool) -> pd.Series:
    """
    OSM tags like lanes/maxspeed can be semicolon-separated ("40;50", "3;2").
    Flags a row bad only if it's non-null and NOT all ';'-parts parse as numbers
    (and, if positive_only, all parts > 0).
    """
    def bad(val):
        if pd.isna(val):
            return False
        parts = str(val).split(";")
        nums = pd.to_numeric(parts, errors="coerce")
        if pd.isna(nums).any():
            return True
        if positive_only and (nums <= 0).any():
            return True
        return False
    return series.apply(bad)
 
 
# --------------------------------------------------------------------------
def check_nodes_osm(nodes: pd.DataFrame, results: list) -> np.ndarray:
    print("=== NODES ===")
    check_schema_osm(nodes, NODE_COLS_OSM, "nodes", results)
 
    _record(results, "no fully duplicate rows", "fail", nodes[nodes.duplicated()])
    _record(results, "node_id is unique", "fail",
            nodes[nodes["node_id"].duplicated(keep=False)].sort_values("node_id"))
    _record(results, "node_id not null", "fail", nodes[nodes["node_id"].isna()])
    _record(results, "geometry not null", "fail", nodes[nodes["geometry"].isna()])
 
    bad_roles = nodes[~nodes["node_role"].isin(VALID_NODE_ROLES_OSM)]
    _record(results, f"node_role in {sorted(VALID_NODE_ROLES_OSM)}", "fail", bad_roles)
 
    geoms = _parse_geometries(nodes["geometry"])
    is_bad = np.array([g is None or g.geom_type != "Point" or not g.is_valid
                        for g in geoms])
    _record(results, "geometry parses as valid POINT", "fail", nodes[is_bad])
    print()
    return geoms
 
 
def check_edges_osm(edges: pd.DataFrame, nodes: pd.DataFrame, node_geoms: np.ndarray,
                 results: list) -> np.ndarray:
    print("=== EDGES ===")
    check_schema_osm(edges, EDGE_COLS_OSM, "edges", results)
 
    _record(results, "no fully duplicate rows", "fail", edges[edges.duplicated()])
    _record(results, "edge_id not null", "fail", edges[edges["edge_id"].isna()])
    _record(results, "from_node_id not null", "fail", edges[edges["from_node_id"].isna()])
    _record(results, "to_node_id not null", "fail", edges[edges["to_node_id"].isna()])
 
    # referential integrity
    node_id_to_geom = dict(zip(nodes["node_id"], node_geoms))
    node_ids = set(node_id_to_geom)
    _record(results, "from_node_id exists in nodes table", "fail",
            edges[~edges["from_node_id"].isin(node_ids)])
    _record(results, "to_node_id exists in nodes table", "fail",
            edges[~edges["to_node_id"].isin(node_ids)])
 
    _record(results, "no self-loop edges (from == to)", "fail",
            edges[edges["from_node_id"] == edges["to_node_id"]])
    _record(results, "no duplicate (edge_id, from_node_id, to_node_id) rows", "warn",
            edges[edges.duplicated(subset=["edge_id", "from_node_id", "to_node_id"],
                                    keep=False)],
            note="often expected for bidirectional OSM ways sharing one edge_id")
 
    # numeric sanity
    _record(results, "length not null", "fail", edges[edges["length"].isna()])
    _record(results, "length is non-negative", "fail", edges[edges["length"] < 0])
    _record(results, "tts (travel time) is non-negative", "fail", edges[edges["tts"] < 0])
 
    _record(results, "oneway is boolean dtype", "fail",
            edges if edges["oneway"].dtype != bool else edges.iloc[0:0])
    _record(results, "reversed is boolean dtype", "fail",
            edges if edges["reversed"].dtype != bool else edges.iloc[0:0])
 
    # geometry: parse once, reuse for validity + endpoint + length checks
    edge_geoms = _parse_geometries(edges["geometry"])
    is_bad_line = np.array([g is None or g.geom_type != "LineString" or not g.is_valid
                             for g in edge_geoms])
    _record(results, "geometry parses as valid LINESTRING", "fail", edges[is_bad_line])
 
    # topology: edge endpoints must match the coordinates of from/to nodes
    ok_geom = ~is_bad_line
    mismatch_mask = np.zeros(len(edges), dtype=bool)
    from_ids = edges["from_node_id"].to_numpy()
    to_ids = edges["to_node_id"].to_numpy()
    for i in np.where(ok_geom)[0]:
        line = edge_geoms[i]
        from_pt = node_id_to_geom.get(from_ids[i])
        to_pt = node_id_to_geom.get(to_ids[i])
        if from_pt is None or to_pt is None:
            continue  # already caught by referential-integrity check above
        coords = list(line.coords)
        start, end = coords[0], coords[-1]
        d_start = ((start[0] - from_pt.x) ** 2 + (start[1] - from_pt.y) ** 2) ** 0.5
        d_end = ((end[0] - to_pt.x) ** 2 + (end[1] - to_pt.y) ** 2) ** 0.5
        if d_start > COORD_TOL or d_end > COORD_TOL:
            mismatch_mask[i] = True
    _record(results, "edge geometry endpoints match node coords", "fail",
            edges[mismatch_mask], note=f"tolerance {COORD_TOL} m")
 
    # length column vs geometry-derived length
    geom_len = np.array([g.length if g is not None else np.nan for g in edge_geoms])
    with np.errstate(invalid="ignore", divide="ignore"):
        rel_diff = np.abs(geom_len - edges["length"].to_numpy()) / geom_len
    len_mismatch = ok_geom & (rel_diff > LENGTH_REL_TOL)
    _record(results, "`length` column matches geometry length", "warn",
            edges[len_mismatch], note=f"tolerance {LENGTH_REL_TOL:.0%}")
 
    # lanes / maxspeed: allow OSM ';'-separated multi-values
    bad_speed = edges[_multi_value_bad_mask(edges["maxspeed"], positive_only=True)]
    _record(results, "maxspeed numeric (incl. ';'-separated values) when present",
            "fail", bad_speed)
 
    bad_lanes = edges[_multi_value_bad_mask(edges["lanes"], positive_only=True)]
    _record(results, "lanes numeric & positive (incl. ';'-separated values) when present",
            "fail", bad_lanes)
 
    # highway tag vocabulary (informational — compound tags are valid OSM usage)
    def unknown_highway(val):
        if pd.isna(val):
            return True
        return not set(str(val).split(";")).issubset(KNOWN_HIGHWAY_TAGS)
    bad_hwy = edges[edges["highway"].apply(unknown_highway)]
    _record(results, "highway tag(s) within known OSM vocabulary", "warn", bad_hwy)
    print()
    return edge_geoms
 
 

def check_graph_consistency_osm(nodes: pd.DataFrame, edges: pd.DataFrame, results: list):
    print("=== CROSS-TABLE CONSISTENCY ===")
    used_ids = set(edges["from_node_id"]) | set(edges["to_node_id"])
    orphan_nodes = nodes[~nodes["node_id"].isin(used_ids)]
    print(f"[INFO] nodes not referenced by any edge: {len(orphan_nodes)}")
 
    # Topological degree: count DISTINCT neighbor connections per node, not raw
    # edge rows. Multiple edges can share the same (from_node_id, to_node_id)
    # pair — e.g. bidirectional OSM ways, or the same physical segment carrying
    # more than one edge_id/route — and those should count as ONE connection,
    # not one per row, or degree gets inflated for nodes with such duplicates.
    node_pairs = edges[["from_node_id", "to_node_id"]].to_numpy()
    undirected_pairs = {tuple(sorted(p)) for p in node_pairs}
    unique_edges = pd.DataFrame(undirected_pairs, columns=["a", "b"])
    degree = pd.concat([unique_edges["a"], unique_edges["b"]]).value_counts()

    endpoint_nodes = nodes.loc[nodes["node_role"] == "endpoint"].copy()
    endpoint_nodes["degree"] = endpoint_nodes["node_id"].map(degree).fillna(0).astype(int)
    hi_degree_endpoints = endpoint_nodes[endpoint_nodes["degree"] > 2].sort_values(
        "degree", ascending=False
    )
    print(f"[INFO] 'endpoint' nodes with degree > 2 (unexpected for a dead-end role, "
        f"degree = count of distinct connected nodes): {len(hi_degree_endpoints)}")

    if len(hi_degree_endpoints):
        print(hi_degree_endpoints[["node_id", "node_role", "degree", "geometry"]]
              .to_string(index=False))
    print()
 
 
 
def run_all_checks_osm(nodes: pd.DataFrame, edges: pd.DataFrame):
    results: list[CheckResult] = []
    node_geoms = check_nodes_osm(nodes, results)
    check_edges_osm(edges, nodes, node_geoms, results)
    check_graph_consistency_osm(nodes, edges, results)
 
    n_fail = sum(1 for r in results if r.level == "fail" and r.n_issues)
    n_warn = sum(1 for r in results if r.level == "warn" and r.n_issues)
    n_ok = sum(1 for r in results if r.n_issues == 0)
    print("=== SUMMARY ===")
    print(f"{n_ok} passed, {n_fail} failed, {n_warn} warned "
          f"(out of {len(results)} checks)")
    #return results


In [4]:

@dataclass
class CheckResult:
    name: str
    level: str          # "fail" | "warn"
    n_issues: int
    issues: pd.DataFrame | None = None
    note: str = ""
 
 
def _fmt(r: CheckResult) -> str:
    tag = "OK" if r.n_issues == 0 else f"{r.level.upper()} ({r.n_issues})"
    return f"[{tag}] {r.name}" + (f" — {r.note}" if r.note else "")
 
 
def _record(results, name, level, subset, note=""):
    r = CheckResult(name=name, level=level, n_issues=len(subset),
                     issues=subset if len(subset) else None, note=note)
    results.append(r)
    print(_fmt(r))
    if r.n_issues and r.issues is not None:
        print(r.issues.head(5).to_string())
        print()
    return r
 
 
def check_schema_gtfs(df, expected_cols, name, results):
    print(f"--- {name}: schema ---")
    missing = set(expected_cols) - set(df.columns)
    extra = set(df.columns) - set(expected_cols)
    _record(results, f"{name}: expected columns present", "fail",
            pd.Index(sorted(missing)))
    if extra:
        print(f"[INFO] {name}: extra/unexpected columns: {sorted(extra)}")
    print()
 
 
def _parse_geometries(geom_series: pd.Series) -> np.ndarray:
    """
    Return a numpy array of shapely geometries from a column that may hold
    either WKT strings (plain pandas DataFrame / CSV read) or already-parsed
    shapely geometry objects (e.g. a GeoDataFrame's geometry column).
    Malformed/missing entries become None.
    """
    values = geom_series.to_numpy()
    is_str = np.array([isinstance(v, str) for v in values])
 
    out = np.empty(len(values), dtype=object)
    if is_str.any():
        str_vals = np.where(is_str, values, "")
        parsed = shapely.from_wkt(str_vals, on_invalid="ignore")
        out[is_str] = parsed[is_str]
 
    not_str = ~is_str
    if not_str.any():
        for i in np.where(not_str)[0]:
            v = values[i]
            out[i] = v if isinstance(v, shapely.Geometry) else None
 
    return out
 
 
def _parse_hourly_string(s) -> dict:
    """'00-01:120|01-02:120|...' -> {'00-01': 120.0, ...}. Empty dict if NaN/malformed."""
    if pd.isna(s) or not str(s):
        return {}
    out = {}
    try:
        for part in str(s).split("|"):
            hour, val = part.split(":")
            out[hour] = float(val)
    except (ValueError, IndexError):
        return {}
    return out
 
 
# --------------------------------------------------------------------------
def check_nodes_gtfs(nodes: pd.DataFrame, results: list) -> np.ndarray:
    print("=== NODES (stops) ===")
    check_schema_gtfs(nodes, NODE_COLS_GTFS, "nodes", results)
 
    _record(results, "no fully duplicate rows", "fail", nodes[nodes.duplicated()])
    _record(results, "node_id is unique", "fail",
            nodes[nodes["node_id"].duplicated(keep=False)].sort_values("node_id"))
    _record(results, "node_id not null", "fail", nodes[nodes["node_id"].isna()])
    _record(results, "stop_name not null", "warn", nodes[nodes["stop_name"].isna()])
    _record(results, "geometry not null", "fail", nodes[nodes["geometry"].isna()])
 
    geoms = _parse_geometries(nodes["geometry"])
    is_bad = np.array([g is None or g.geom_type != "Point" or not g.is_valid
                        for g in geoms])
    _record(results, "geometry parses as valid POINT", "fail", nodes[is_bad])
    print()
    return geoms
 
 
def check_edges_gtfs(edges: pd.DataFrame, nodes: pd.DataFrame, node_geoms: np.ndarray,
                 results: list) -> np.ndarray:
    print("=== EDGES (route segments) ===")
    check_schema_gtfs(edges, EDGE_COLS_GTFS, "edges", results)
 
    _record(results, "edge_id not null", "fail", edges[edges["edge_id"].isna()])
    _record(results, "from_node_id not null", "fail", edges[edges["from_node_id"].isna()])
    _record(results, "to_node_id not null", "fail", edges[edges["to_node_id"].isna()])
    _record(results, "route_id not null", "fail", edges[edges["route_id"].isna()])
 
    # referential integrity — NOTE: `nodes` may be a stop subset that doesn't
    # include every stop referenced by a route passing through the area, so
    # this is a warning rather than a hard failure.
    node_id_to_geom = dict(zip(nodes["node_id"], node_geoms))
    node_ids = set(node_id_to_geom)
    _record(results, "from_node_id exists in nodes table", "warn",
            edges[~edges["from_node_id"].isin(node_ids)],
            note="expected if nodes.csv is a spatial subset of stops")
    _record(results, "to_node_id exists in nodes table", "warn",
            edges[~edges["to_node_id"].isin(node_ids)],
            note="expected if nodes.csv is a spatial subset of stops")
 
    _record(results, "no self-loop edges (from == to)", "fail",
            edges[edges["from_node_id"] == edges["to_node_id"]])
 
    # geometry validity
    edge_geoms = _parse_geometries(edges["geometry"])
    is_bad_line = np.array([g is None or g.geom_type != "LineString" or not g.is_valid
                             for g in edge_geoms])
    _record(results, "geometry parses as valid LINESTRING", "fail", edges[is_bad_line])
 
    # topology: edge endpoints should match the coordinates of from/to nodes
    # (only checkable for edges whose endpoints are present in the nodes table)
    ok_geom = ~is_bad_line
    mismatch_mask = np.zeros(len(edges), dtype=bool)
    from_ids = edges["from_node_id"].to_numpy()
    to_ids = edges["to_node_id"].to_numpy()
    for i in np.where(ok_geom)[0]:
        from_pt = node_id_to_geom.get(from_ids[i])
        to_pt = node_id_to_geom.get(to_ids[i])
        if from_pt is None or to_pt is None:
            continue
        coords = list(edge_geoms[i].coords)
        start, end = coords[0], coords[-1]
        d_start = ((start[0] - from_pt.x) ** 2 + (start[1] - from_pt.y) ** 2) ** 0.5
        d_end = ((end[0] - to_pt.x) ** 2 + (end[1] - to_pt.y) ** 2) ** 0.5
        if d_start > COORD_TOL or d_end > COORD_TOL:
            mismatch_mask[i] = True
    _record(results, "edge geometry endpoints match node coords", "fail",
            edges[mismatch_mask], note=f"tolerance {COORD_TOL} m, only checked "
                                        "where both endpoints exist in nodes")
 
    # trip_count / tts sanity
    _record(results, "trip_count is positive", "fail", edges[edges["trip_count"] <= 0])
    _record(results, "tts_mean is non-negative", "fail", edges[edges["tts_mean"] < 0])
    _record(results, "tts_min <= tts_mean <= tts_max", "fail",
            edges[~((edges["tts_min"] <= edges["tts_mean"]) &
                    (edges["tts_mean"] <= edges["tts_max"]))])
 
    # trip_count should equal the sum of its peak_am/peak_pm/rest_of_day breakdown
    parts = edges[["trip_count_peak_am", "trip_count_peak_pm",
                   "trip_count_rest_of_day"]].fillna(0)
    calc = parts.sum(axis=1)
    _record(results, "trip_count == peak_am + peak_pm + rest_of_day breakdown",
            "fail", edges[calc != edges["trip_count"]])
 
    # trip_count should equal the sum of the hourly_trip_counts string
    hourly_sum = edges["hourly_trip_counts"].apply(
        lambda s: sum(_parse_hourly_string(s).values()))
    _record(results, "trip_count == sum(hourly_trip_counts)", "fail",
            edges[hourly_sum != edges["trip_count"]])
 
    # hourly_travel_times / hourly_trip_counts parse cleanly and share the same hour keys
    def bad_hourly_pair(row):
        tt = _parse_hourly_string(row["hourly_travel_times"])
        tc = _parse_hourly_string(row["hourly_trip_counts"])
        return (not tt) or (not tc) or (set(tt) != set(tc))
    bad_hourly = edges[edges.apply(bad_hourly_pair, axis=1)]
    _record(results, "hourly_travel_times / hourly_trip_counts parse & align",
            "fail", bad_hourly)
 
    # days_of_week_summary uses known day names only
    def bad_days(val):
        if pd.isna(val):
            return True
        return not set(str(val).split("|")).issubset(VALID_DAYS)
    _record(results, "days_of_week_summary uses valid day names", "fail",
            edges[edges["days_of_week_summary"].apply(bad_days)])

    print()
    return edge_geoms
 
 
def check_route_consistency_gtfs(edges: pd.DataFrame, results: list):
    print("=== ROUTE-LEVEL CONSISTENCY ===")
    # a route_id should map to exactly one route_short_name / route_long_name
    grp = edges.groupby("route_id")[["route_short_name", "route_long_name"]].nunique()
    inconsistent_routes = grp[(grp["route_short_name"] > 1) | (grp["route_long_name"] > 1)]
    print(f"[INFO] route_id values with inconsistent short/long name across rows: "
          f"{len(inconsistent_routes)}")
    if len(inconsistent_routes):
        print(inconsistent_routes.to_string())
    print()
 
    # edges per route_id repeated across multiple (from,to) segments (normal for
    # a route's full stop sequence) — just surface the distribution
    seg_counts = edges.groupby("route_id").size()
    print(f"[INFO] route_id count: {edges['route_id'].nunique()}; "
          f"segments per route — min {seg_counts.min()}, "
          f"median {int(seg_counts.median())}, max {seg_counts.max()}")
    print()
 
 
def run_all_checks_gtfs(nodes: pd.DataFrame, edges: pd.DataFrame):
    results: list[CheckResult] = []
    node_geoms = check_nodes_gtfs(nodes, results)
    check_edges_gtfs(edges, nodes, node_geoms, results)
    check_route_consistency_gtfs(edges, results)
 
    n_fail = sum(1 for r in results if r.level == "fail" and r.n_issues)
    n_warn = sum(1 for r in results if r.level == "warn" and r.n_issues)
    n_ok = sum(1 for r in results if r.n_issues == 0)
    print("=== SUMMARY ===")
    print(f"{n_ok} passed, {n_fail} failed, {n_warn} warned "
          f"(out of {len(results)} checks)")
    #return results

# GeoJSON Test

In [5]:
main(
    area_name="Valencia",
    modes=["train", "bus", "driving", "bike", "walking"],
    output_path=output_folder,
    crs="epsg:25830",
    output_file_type="geojson",
    boundaries_path= "/Users/aidavillalba/Desktop/descargas_multimodalpy/input_shapes/capa_Valencia.shp",
    api_key=NAP_API_KEY
)

[aviso] No se pudo normalizar el feed 20260727_080002_BUS_Samar.zip: 'stop_lat'
[aviso] No se pudo normalizar el feed 20260731_000019_BUS_Cantabria.zip: Python int too large to convert to C long
[aviso] No se pudo normalizar el feed 20260825_040012_RENFE_CERCA.zip: 'stop_sequence'
{
  "area_name": "Valencia",
  "modes": [
    "train",
    "bus",
    "driving",
    "bike",
    "walking"
  ],
  "crs": "epsg:25830",
  "output_file_type": "geojson",
  "output_path": "/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia",
  "osm_modes": [
    "driving",
    "bike",
    "walking"
  ],
  "gtfs_datasets": [
    "20260825_000044_bus_alsa",
    "20260825_020032_genvalenciana_interurbano",
    "20260731_200004_emt_valencia",
    "20260821_140005_metro_valencia",
    "20260702_060101_ouigo",
    "20260825_000005_renfe_avld"
  ],
  "files": [
    "study_area_boundary.geojson",
    "osm_driving_nodes.geojson",
    "osm_driving_edges.geojson",
    "osm_bike_nodes.geojson",
    "osm_bike

{'area_name': 'Valencia',
 'modes': ['train', 'bus', 'driving', 'bike', 'walking'],
 'crs': 'epsg:25830',
 'output_file_type': 'geojson',
 'output_path': '/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia',
 'osm_modes': ['driving', 'bike', 'walking'],
 'gtfs_datasets': ['20260825_000044_bus_alsa',
  '20260825_020032_genvalenciana_interurbano',
  '20260731_200004_emt_valencia',
  '20260821_140005_metro_valencia',
  '20260702_060101_ouigo',
  '20260825_000005_renfe_avld'],
 'files': ['study_area_boundary.geojson',
  'osm_driving_nodes.geojson',
  'osm_driving_edges.geojson',
  'osm_bike_nodes.geojson',
  'osm_bike_edges.geojson',
  'osm_walking_nodes.geojson',
  'osm_walking_edges.geojson',
  'gtfs_20260825_000044_bus_alsa_nodes.geojson',
  'gtfs_20260825_000044_bus_alsa_edges.geojson',
  'gtfs_20260825_020032_genvalenciana_interurbano_nodes.geojson',
  'gtfs_20260825_020032_genvalenciana_interurbano_edges.geojson',
  'gtfs_20260731_200004_emt_valencia_nodes.geojson',
 

In [14]:
#Driving
df_driving_nodes = gpd.read_file(f"{output_folder}/osm_driving_nodes.geojson")
df_driving_edges = gpd.read_file(f"{output_folder}/osm_driving_edges.geojson")

#Walking
df_walking_nodes = gpd.read_file(f"{output_folder}/osm_walking_nodes.geojson")
df_walking_edges = gpd.read_file(f"{output_folder}/osm_walking_edges.geojson")

#Bike
df_bike_nodes = gpd.read_file(f"{output_folder}/osm_bike_nodes.geojson")
df_bike_edges = gpd.read_file(f"{output_folder}/osm_bike_edges.geojson")

#Train
df_train_nodes = gpd.read_file(f"{output_folder}/gtfs_20260813_140007_metro_valencia_nodes.geojson")
df_train_edges = gpd.read_file(f"{output_folder}/gtfs_20260813_140007_metro_valencia_edges.geojson")

#Bus
df_bus_nodes = gpd.read_file(f"{output_folder}/gtfs_20260815_020044_genvalenciana_interurbano_nodes.geojson")
df_bus_edges = gpd.read_file(f"{output_folder}/gtfs_20260815_020044_genvalenciana_interurbano_edges.geojson")

### Driving

In [15]:
run_all_checks_osm(df_driving_nodes, df_driving_edges)

=== NODES ===
--- nodes: schema ---
[OK] nodes: expected columns present

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] geometry not null
[OK] node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']
[OK] geometry parses as valid POINT

=== EDGES ===
--- edges: schema ---
[OK] edges: expected columns present

[OK] no fully duplicate rows
[OK] edge_id not null
[OK] from_node_id not null
[OK] to_node_id not null
[OK] from_node_id exists in nodes table
[OK] to_node_id exists in nodes table
[OK] no self-loop edges (from == to)
[OK] no duplicate (edge_id, from_node_id, to_node_id) rows — often expected for bidirectional OSM ways sharing one edge_id
[OK] length not null
[OK] length is non-negative
[OK] tts (travel time) is non-negative
[OK] oneway is boolean dtype
[OK] reversed is boolean dtype
[OK] geometry parses as valid LINESTRING
[OK] edge geometry endpoints match node coords — tolerance 0.01 m
[OK] `length` column matches geometry 

### Walking


In [16]:
run_all_checks_osm(df_walking_nodes, df_walking_edges)

=== NODES ===
--- nodes: schema ---
[OK] nodes: expected columns present

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] geometry not null
[OK] node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']
[OK] geometry parses as valid POINT

=== EDGES ===
--- edges: schema ---
[OK] edges: expected columns present

[OK] no fully duplicate rows
[OK] edge_id not null
[OK] from_node_id not null
[OK] to_node_id not null
[OK] from_node_id exists in nodes table
[OK] to_node_id exists in nodes table
[OK] no self-loop edges (from == to)
[OK] no duplicate (edge_id, from_node_id, to_node_id) rows — often expected for bidirectional OSM ways sharing one edge_id
[OK] length not null
[OK] length is non-negative
[OK] tts (travel time) is non-negative
[OK] oneway is boolean dtype
[OK] reversed is boolean dtype
[OK] geometry parses as valid LINESTRING
[OK] edge geometry endpoints match node coords — tolerance 0.01 m
[OK] `length` column matches geometry 

### Bike

In [17]:
run_all_checks_osm(df_bike_nodes, df_bike_edges)

=== NODES ===
--- nodes: schema ---
[OK] nodes: expected columns present

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] geometry not null
[OK] node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']
[OK] geometry parses as valid POINT

=== EDGES ===
--- edges: schema ---
[OK] edges: expected columns present

[OK] no fully duplicate rows
[OK] edge_id not null
[OK] from_node_id not null
[OK] to_node_id not null
[OK] from_node_id exists in nodes table
[OK] to_node_id exists in nodes table
[OK] no self-loop edges (from == to)
[OK] no duplicate (edge_id, from_node_id, to_node_id) rows — often expected for bidirectional OSM ways sharing one edge_id
[OK] length not null
[OK] length is non-negative
[OK] tts (travel time) is non-negative
[OK] oneway is boolean dtype
[OK] reversed is boolean dtype
[OK] geometry parses as valid LINESTRING
[OK] edge geometry endpoints match node coords — tolerance 0.01 m
[OK] `length` column matches geometry 

### Bus

In [18]:
run_all_checks_gtfs(df_bus_nodes, df_bus_edges)

=== NODES (stops) ===
--- nodes: schema ---
[OK] nodes: expected columns present

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] stop_name not null
[OK] geometry not null
[OK] geometry parses as valid POINT

=== EDGES (route segments) ===
--- edges: schema ---
[OK] edges: expected columns present

[OK] edge_id not null
[OK] from_node_id not null
[OK] to_node_id not null
[OK] route_id not null
[WARN (125)] from_node_id exists in nodes table — expected if nodes.csv is a spatial subset of stops
                                                             edge_id from_node_id    to_node_id    route_id route_short_name                                      route_long_name  trip_count  tts_mean  tts_min  tts_max  tts_mean_peak_am  tts_mean_peak_pm  tts_mean_rest_of_day  trip_count_peak_am  trip_count_peak_pm  trip_count_rest_of_day  tts_mean_weekday  trip_count_weekday  tts_mean_weekend  trip_count_weekend                      days_of_week_summary              

### Train

In [19]:
run_all_checks_gtfs(df_train_nodes, df_train_edges)

=== NODES (stops) ===
--- nodes: schema ---
[OK] nodes: expected columns present

[OK] no fully duplicate rows
[OK] node_id is unique
[OK] node_id not null
[OK] stop_name not null
[OK] geometry not null
[OK] geometry parses as valid POINT

=== EDGES (route segments) ===
--- edges: schema ---
[OK] edges: expected columns present

[OK] edge_id not null
[OK] from_node_id not null
[OK] to_node_id not null
[OK] route_id not null
[WARN (153)] from_node_id exists in nodes table — expected if nodes.csv is a spatial subset of stops
                                  edge_id from_node_id to_node_id  route_id route_short_name                route_long_name  trip_count  tts_mean  tts_min  tts_max  tts_mean_peak_am  tts_mean_peak_pm  tts_mean_rest_of_day  trip_count_peak_am  trip_count_peak_pm  trip_count_rest_of_day  tts_mean_weekday  trip_count_weekday  tts_mean_weekend  trip_count_weekend                                      days_of_week_summary                                                    

# Networkx

### Shortest path calculation

In [ ]:
import json
import networkx as nx
import geopandas as gpd

In [ ]:
"""with open("/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia/osm_driving_graph.json", "r") as f:
    data = json.load(f)

G = nx.node_link_graph(data)"""

In [ ]:
edges_df = gpd.read_file("/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia/osm_driving_edges.geojson")

G = nx.from_pandas_edgelist(
    edges_df,
    source="from_node_id",
    target="to_node_id",
    edge_attr=True,
    create_using=nx.DiGraph()
)

In [ ]:
weighted_path = nx.shortest_path(G, source="driving_topology_node_0024046", target="driving_topology_node_0006196", weight="tts")


In [ ]:
edges_df[edges_df["from_node_id"].isin(weighted_path)].to_file("/Users/aidavillalba/Desktop/descargas_multimodalpy/shortest_paths/shortest_path1.geojson")